# RAG Evaluation Results — SpirulinaAI

**Evaluation type**: LLM-as-judge (llama-3.3-70b-versatile via Groq)  
**Metrics**:
- **Answer Relevance** — does the answer address the question? (target > 0.80)
- **Faithfulness** — are all claims grounded in the retrieved context? (target > 0.85)
- **Context Recall** — do retrieved chunks contain the key concepts? (keyword-based, target > 0.70)

**Test set**: 30 questions — 10 factual, 10 troubleshooting, 10 operational  
**Retrieval**: ChromaDB + paraphrase-multilingual-MiniLM-L12-v2, top-k=5


## 1. Setup

In [ ]:
import sys, os, json
from pathlib import Path

# Make sure project root is on the path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

import pandas as pd
import numpy as np

RESULTS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'rag_eval_results.json'
print('Project root:', PROJECT_ROOT)
print('Results path:', RESULTS_PATH)
print('Results exist:', RESULTS_PATH.exists())

## 2. Run Evaluation (or load cached results)

Run the cell below **once** — it takes ~5-10 minutes (30 generate + 30 judge LLM calls).  
Re-runs load from the cached JSON.


In [ ]:
FORCE_RERUN = False   # Set True to re-run even if results exist
TOP_K = 5             # Retrieval top-k (change to 8 for tuned run)

if RESULTS_PATH.exists() and not FORCE_RERUN:
    print(f'Loading cached results from {RESULTS_PATH}')
    with open(RESULTS_PATH, encoding='utf-8') as f:
        output = json.load(f)
    print(f"Config: top_k={output['eval_config']['top_k']}, "
          f"questions={output['eval_config']['num_questions']}")
else:
    from tests.rag_eval import run_eval
    output = run_eval(top_k=TOP_K, verbose=True)

summary = output['summary']
results = output['results']
print('\nEvaluation loaded successfully.')

## 3. Overall Scores vs. Targets

In [ ]:
TARGETS = {'answer_relevance': 0.80, 'faithfulness': 0.85, 'context_recall': 0.70}
overall = summary['overall']

rows = []
for metric, target in TARGETS.items():
    score = overall[metric]
    passed = score is not None and score >= target
    rows.append({
        'Metric':  metric.replace('_', ' ').title(),
        'Score':   round(score, 3) if score is not None else 'N/A',
        'Target':  target,
        'Delta':   round(score - target, 3) if score is not None else 'N/A',
        'Status':  'PASS' if passed else 'FAIL',
    })

df_overall = pd.DataFrame(rows)
print('\n=== OVERALL RESULTS ===')
print(df_overall.to_string(index=False))

all_pass = all(summary['pass'].values())
verdict = 'ALL METRICS PASS - Pipeline is production-ready!' if all_pass else 'SOME METRICS FAIL - See tuning recommendations below.'
print(f'\nVerdict: {verdict}')

## 4. Scores by Category

In [ ]:
rows = []
for cat, scores in summary['per_category'].items():
    rows.append({
        'Category':         cat.title(),
        'Answer Relevance': round(scores['answer_relevance'], 3) if scores['answer_relevance'] else 'N/A',
        'Faithfulness':     round(scores['faithfulness'],     3) if scores['faithfulness']     else 'N/A',
        'Context Recall':   round(scores['context_recall'],   3) if scores['context_recall']   else 'N/A',
    })

df_cat = pd.DataFrame(rows)
print('\n=== SCORES BY CATEGORY ===')
print(df_cat.to_string(index=False))

## 5. Per-Question Breakdown

In [ ]:
df = pd.DataFrame(results)

# Format for display
df_display = df[['id', 'category', 'question', 'context_recall', 'answer_relevance', 'faithfulness', 'reasoning']].copy()
df_display['question'] = df_display['question'].str[:65] + '...'
df_display.columns = ['ID', 'Category', 'Question', 'Recall', 'Relevance', 'Faithful', 'Judge Reasoning']

# Flag failing rows
def flag(row):
    flags = []
    if row['Relevance'] is not None and row['Relevance'] < TARGETS['answer_relevance']:
        flags.append('low-relevance')
    if row['Faithful'] is not None and row['Faithful'] < TARGETS['faithfulness']:
        flags.append('low-faithful')
    if row['Recall'] is not None and row['Recall'] < TARGETS['context_recall']:
        flags.append('low-recall')
    return ', '.join(flags) if flags else 'OK'

df_display['Flag'] = df_display.apply(flag, axis=1)

print('\n=== PER-QUESTION RESULTS ===')
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 160)
print(df_display[['ID', 'Category', 'Question', 'Recall', 'Relevance', 'Faithful', 'Flag']].to_string(index=False))

## 6. Failing Questions Deep-Dive

In [ ]:
# Show full answers for questions that failed any metric
failing = [r for r in results if (
    (r['answer_relevance'] is not None and r['answer_relevance'] < TARGETS['answer_relevance']) or
    (r['faithfulness']     is not None and r['faithfulness']     < TARGETS['faithfulness'])     or
    (r['context_recall']                < TARGETS['context_recall'])
)]

if not failing:
    print('No failing questions — all metrics pass on every question!')
else:
    print(f'{len(failing)} question(s) below target in at least one metric:\n')
    for r in failing:
        print(f"[Q{r['id']}] {r['category'].upper()} | recall={r['context_recall']:.2f} "
              f"relevance={r['answer_relevance']} faithful={r['faithfulness']}")
        print(f"  Question : {r['question']}")
        print(f"  Reasoning: {r['reasoning']}")
        if r['answer']:
            print(f"  Answer   : {r['answer'][:200]}..." if len(r['answer']) > 200 else f"  Answer   : {r['answer']}")
        print()

## 7. Score Distribution

In [ ]:
# Text-based score distribution (no matplotlib needed)
def text_histogram(values, label, bins=5):
    values = [v for v in values if v is not None]
    if not values:
        print(f'{label}: no data')
        return
    min_v, max_v = 0.0, 1.0
    step = (max_v - min_v) / bins
    counts = [0] * bins
    for v in values:
        idx = min(int((v - min_v) / step), bins - 1)
        counts[idx] += 1
    print(f'\n{label} distribution (n={len(values)}, mean={np.mean(values):.3f})')
    for i, c in enumerate(counts):
        lo = min_v + i * step
        hi = lo + step
        bar = '#' * (c * 3)
        print(f'  {lo:.1f}-{hi:.1f} | {bar} {c}')

ar_vals = [r['answer_relevance'] for r in results]
fa_vals = [r['faithfulness']     for r in results]
cr_vals = [r['context_recall']   for r in results]

text_histogram(ar_vals, 'Answer Relevance')
text_histogram(fa_vals, 'Faithfulness')
text_histogram(cr_vals, 'Context Recall')

## 8. Tuning Recommendations

In [ ]:
passes = summary['pass']

if all(passes.values()):
    print('All metrics pass targets.')
    print('The RAG pipeline is production-ready.')
else:
    print('=== TUNING RECOMMENDATIONS ===')

    if not passes.get('context_recall', True):
        print('''
Context Recall below 0.70:
  1. Run eval with --top-k 8  (retrieve 8 chunks instead of 5)
  2. Re-ingest with chunk_size=300 for finer-grained chunks
  3. Check that PDFs are in topic subfolders (papers/, manuals/)
     for proper metadata tagging and filtered retrieval
''')

    if not passes.get('faithfulness', True):
        print('''
Faithfulness below 0.85:
  1. Reduce generator temperature: 0.2 -> 0.0 in rag/generator/generate.py
  2. Strengthen RULE 1 in system prompt:
     Add: "If a fact is not in the context, say so — do not infer."
  3. Add source citation reminder to _HUMAN_TEMPLATE:
     "Ground your answer strictly in the <context> above."
  4. Increase top_k to 8 for more context coverage
''')

    if not passes.get('answer_relevance', True):
        print('''
Answer Relevance below 0.80:
  1. Increase top_k to give the LLM more context to work with
  2. Check whether knowledge base covers the failing question topics
  3. Add topic-filtered retrieval for category-specific queries
  4. Verify ChromaDB has enough chunks (target: 500+ clean chunks)
''')

## 9. Context Recall Only (Fast Check — No LLM Calls)

In [ ]:
# Run context recall only — fast, no API calls
# Useful to verify retrieval quality without burning Groq quota

RUN_FAST_CHECK = False  # Set to True to re-check retrieval now

if RUN_FAST_CHECK:
    from tests.rag_eval import run_eval
    fast_output = run_eval(
        top_k=TOP_K,
        skip_generate=True,
        save_path='data/processed/rag_eval_recall_only.json',
        verbose=True,
    )
    fast_recall = fast_output['summary']['overall']['context_recall']
    print(f'\nContext recall (top_k={TOP_K}): {fast_recall:.3f}')
else:
    print('Set RUN_FAST_CHECK = True to run retrieval-only evaluation.')

## 10. Export Summary Table

In [ ]:
# Export a clean CSV for reporting
df_export = pd.DataFrame(results)[['id','category','question','context_recall','answer_relevance','faithfulness','reasoning']]
csv_path = PROJECT_ROOT / 'data' / 'processed' / 'rag_eval_results.csv'
df_export.to_csv(csv_path, index=False, encoding='utf-8')
print(f'CSV exported to {csv_path}')
print(f'\nFinal scores:')
print(f'  Answer Relevance : {overall["answer_relevance"]:.3f}  (target 0.80)')
print(f'  Faithfulness     : {overall["faithfulness"]:.3f}  (target 0.85)')
print(f'  Context Recall   : {overall["context_recall"]:.3f}  (target 0.70)')